In [ ]:
# Fabric-native parameters and config
# Edit these values directly in Fabric, or pass them from a Fabric Data Pipeline.
ENV = "dev"
SCHEMA = "dbo"
MARKET = "AU"
MOCK_IPSTACK = True
LOOKBACK_DAYS = 90
LISTINGS_FILE = "listings.csv"
VIEWS_FILE = "property_views_5k.csv"
MAX_IPSTACK_CALLS = 5  # safety cap for live IPstack tests
FORCE_REFRESH_IPSTACK = False
IPSTACK_ACCESS_KEY = ""  # leave blank in git; use Fabric variable/secret or edit in workspace only

try:
    from notebookutils import mssparkutils
except ImportError:
    mssparkutils = None

from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

class Config:
    def __init__(self):
        self.env = ENV
        self.schema = SCHEMA
        self.market = MARKET
        self.mock_ipstack = MOCK_IPSTACK
        self.lookback_days = LOOKBACK_DAYS
        self.listings_file = LISTINGS_FILE
        self.views_file = VIEWS_FILE
        self.max_ipstack_calls = MAX_IPSTACK_CALLS
        self.force_refresh_ipstack = FORCE_REFRESH_IPSTACK
        self.ipstack_access_key = IPSTACK_ACCESS_KEY
        self.files_base = "Files"  # Fabric attached Lakehouse path
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"

        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.schema}.{table}"

conf = Config()
print(f"env={conf.env} market={conf.market} mock={conf.mock_ipstack} views={conf.views_file}")
print(f"listings_path={conf.listings_path}")
print(f"views_path={conf.views_path}")




In [ ]:
import json, os, time, urllib.parse, urllib.request
from pathlib import Path
from pyspark.sql import functions as F

class SilverLoader:
    def __init__(self):
        self.c = conf

    def distinct_ips(self):
        return [r.ip for r in spark.table(self.c.table_fqn(self.c.bronze_property_views)).select("ip").distinct().collect() if r.ip]

    def already_enriched_ips(self):
        if self.c.force_refresh_ipstack:
            return set()
        try:
            if not spark.catalog.tableExists(self.c.table_fqn(self.c.silver_ip_dim)):
                return set()
            return {r.ip for r in spark.table(self.c.table_fqn(self.c.silver_ip_dim)).select("ip").distinct().collect() if r.ip}
        except Exception:
            return set()

    def ips_to_enrich(self, ips):
        existing = self.already_enriched_ips()
        pending = [ip for ip in ips if ip not in existing]
        if self.c.mock_ipstack:
            return pending
        capped = pending[: self.c.max_ipstack_calls]
        skipped = len(pending) - len(capped)
        if skipped > 0:
            print(f"Live IPstack safety cap: enriching {len(capped)} IPs, skipping {skipped}. Increase MAX_IPSTACK_CALLS when ready.")
        return capped

    def load_mock(self, ip):
        for p in [Path("data/fixtures/ipstack")/f"{ip}.json", Path(f"{self.c.fixture_base}/{ip}.json")]:
            if p.exists():
                return p.read_text(), 200
        return json.dumps({"ip": ip, "country_code": "AU", "country_name": "Australia", "region_name": "Victoria", "city": "Melbourne", "connection": {"isp": "Unknown"}, "security": {"is_proxy": False}}), 200

    def ipstack_key(self):
        return self.c.ipstack_access_key or os.getenv("IPSTACK_ACCESS_KEY", "")

    def load_live(self, ip):
        key = self.ipstack_key()
        if not key:
            raise ValueError("IPSTACK_ACCESS_KEY required when MOCK_IPSTACK=False")
        safe_ip = urllib.parse.quote(ip, safe="")
        safe_key = urllib.parse.quote(key, safe="")
        url = f"http://api.ipstack.com/{safe_ip}?access_key={safe_key}&security=1"
        with urllib.request.urlopen(url, timeout=15) as resp:
            return resp.read().decode(), resp.getcode()

    def write_ipstack_error(self, ip, error_message, status=0):
        err = spark.createDataFrame([(ip, str(error_message), int(status))], "ip STRING, error_message STRING, http_status INT")
        err.withColumn("api_called_at", F.current_timestamp()).write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_errors))

    def enrich_ips(self, ips):
        target_ips = self.ips_to_enrich(ips)
        print(f"IPstack target IPs: {len(target_ips)} (distinct={len(ips)}, mock={self.c.mock_ipstack}, force_refresh={self.c.force_refresh_ipstack})")
        if not target_ips:
            return

        rows = []
        for ip in target_ips:
            try:
                body, status = self.load_mock(ip) if self.c.mock_ipstack else self.load_live(ip)
                if not self.c.mock_ipstack:
                    time.sleep(0.25)
                rows.append((ip, body, int(status)))
            except Exception as e:
                self.write_ipstack_error(ip, e)

        if rows:
            df = spark.createDataFrame(rows, "ip STRING, response_json STRING, http_status INT")
            df.withColumn("api_called_at", F.current_timestamp()).write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_raw))

    def build_ip_dim(self):
        j = "ip STRING, country_code STRING, country_name STRING, region_name STRING, city STRING, latitude DOUBLE, longitude DOUBLE, time_zone STRUCT<id:STRING>, connection STRUCT<isp:STRING>, security STRUCT<is_proxy:BOOLEAN,proxy_type:STRING>"
        raw = spark.table(self.c.table_fqn(self.c.bronze_ipstack_raw))
        p = raw.withColumn("j", F.from_json("response_json", j))
        # Map AU region to state abbreviation for interstate analytics
        dim = p.select(
            "ip",
            F.col("j.country_code").alias("country_code"),
            F.col("j.country_name").alias("country_name"),
            F.col("j.region_name").alias("visitor_region"),
            F.col("j.city").alias("visitor_city"),
            F.col("j.latitude").alias("latitude"),
            F.col("j.longitude").alias("longitude"),
            F.col("j.time_zone.id").alias("timezone_id"),
            F.col("j.connection.isp").alias("isp"),
            F.when(F.col("j.security.proxy_type") == "vpn", True).otherwise(F.coalesce(F.col("j.security.is_proxy"), F.lit(False))).alias("is_vpn"),
            F.current_timestamp().alias("enriched_at"),
        ).dropDuplicates(["ip"])
        dim.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_ip_dim))

    def build_listings(self):
        spark.table(self.c.table_fqn(self.c.bronze_listings)).drop("_ingested_at").write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_listings))

    def build_visits(self):
        v = spark.table(self.c.table_fqn(self.c.bronze_property_views))
        l = spark.table(self.c.table_fqn(self.c.silver_listings))
        g = spark.table(self.c.table_fqn(self.c.silver_ip_dim))
        # Derive visitor_state from AU region names
        state_map = (
            F.when(F.col("visitor_region").contains("New South Wales"), "NSW")
            .when(F.col("visitor_region").contains("Victoria"), "VIC")
            .when(F.col("visitor_region").contains("Queensland"), "QLD")
            .when(F.col("visitor_region").contains("South Australia"), "SA")
            .when(F.col("visitor_region").contains("Western Australia"), "WA")
            .otherwise(F.lit("OTHER"))
        )
        device = F.when(F.col("user_agent").contains("iPhone"), F.lit("mobile")).otherwise(F.lit("desktop"))
        out = (v.join(l, "property_id", "inner")
               .join(g, "ip", "left")
               .withColumn("visitor_state", state_map)
               .withColumn("device_type", device)
               .withColumnRenamed("state", "listing_state")
               .select("event_id","session_id","event_ts","user_id","ip","property_id",
                       "view_duration_sec","enquiry_flag","favorite_flag","event_date",
                       F.col("country_name").alias("visitor_country"),
                       "visitor_region","visitor_city","visitor_state","timezone_id","device_type",
                       "suburb","listing_state","property_type","price_range","is_luxury","price"))
        out.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(self.c.table_fqn(self.c.silver_visits_enriched))

    def run(self):
        ips = self.distinct_ips()
        print(f"Enriching {len(ips)} visitor IPs (mock={self.c.mock_ipstack})")
        self.enrich_ips(ips)
        self.build_ip_dim()
        self.build_listings()
        self.build_visits()
        print("✅ Silver complete")

SilverLoader().run()



